In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imodex_miri_postprocess_pipeline as imp #homemade pipeline
from astropy.io import fits
import glob
import copy
import scipy.ndimage as nd
from scipy.optimize import minimize


In [ ]:
path_d = "/home/cperrot/Work/raw/JWST/Target/MAST/CALINTS/P03662/"
path_s = "/home/cperrot/Work/projet/JWST/Target/GO/P3662/calints/"
newd = 201
Y, X = np.ogrid[:newd, :newd]

lam = 11.40e-6 #µm
x0  = 119.77
y0  = 112.16

pixscale = 0.110
scale = [(newd/2.)*pixscale,-1*(newd/2.)*pixscale,-1*(newd/2.)*pixscale,(newd/2.)*pixscale]

dr = 1.*lam/6.5*180.*3600./np.pi/pixscale
nbpixel = np.pi*dr**2

In [47]:
path_d = "/home/cperrot/Work/raw/JWST/Target/MAST/CAL/"
### librairie de référence
#[PID, Obs id, Ref Obs id]
ref_id = [["01194", "015", "019"],
          ["01277", "008", "011"],
          ["01668", "004", "005"],
          ["01668", "009", "010"],
          ["01668", "014", "015"],
          ["02538", "009", "010"],
          ["02538", "004", "005"],
          ["03662", "009", "010"],
          ["01241", "008", "011"],
          ["01241", "020", "023"],
          ["01241", "030", "032"],
          ["01241", "040", "043"],
          ["01386", "006", "029"],
          ["01386", "024", "035"],
          ["01413", "008", "011"],
          ["02153", "005", "004"],
          ["02538", "014", "015"],
          ["02538", "114", "115"],
          ["03254", "008", "007"],
          ["03254", "016", "015"],
          ["03254", "020", "019"]]

cube_ref = []
cpt = 1
for i in ref_id:
    print("reference #",cpt)
    tmp_r = glob.glob(path_d+"P"+i[0]+"/obs"+i[1]+"/*_cal.fits")
    tmp_b = glob.glob(path_d+"P"+i[0]+"/obs"+i[2]+"/*_cal.fits")
    bkg = np.minimum(fits.open(tmp_b[0])[1].data, fits.open(tmp_b[1])[1].data)
    
    for j in tmp_r: 
        tmp = imp.clean_image_nan(np.array(imp.remove_bad_pixel(fits.open(j)[1].data - bkg,3,5)))
        #tmp = imp.clean_image_nan(fits.open(j)[1].data - bkg)
        tmp = nd.shift(tmp,[newd/2-y0,newd/2-x0])[0:newd,0:newd]
        cube_ref.append(tmp)
    cpt += 1
cube_ref = np.array(cube_ref)



reference # 1


/home/cperrot/Work/code/JWST/Module/imodex_miri_postprocess_pipeline.py:250: RuntimeWarning: Mean of empty slice
  function=lambda x: np.nanmean(x[mask.flatten().astype(bool)]),
/home/cperrot/anaconda3/envs/jwst/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


reference # 2
reference # 3
reference # 4
reference # 5
reference # 6
reference # 7
reference # 8
reference # 9
reference # 10
reference # 11
reference # 12
reference # 13
reference # 14
reference # 15
reference # 16
reference # 17
reference # 18
reference # 19
reference # 20
reference # 21


In [48]:
imp.save_fits(cube_ref  , path_s+"cube_lib_ref_1140.fits")


In [53]:
path_d = "/home/cperrot/Work/raw/JWST/Target/MAST/CALINTS/P03662/"

## data loading
sci1_fi    = glob.glob(path_d+"obs001/*_calints.fits")
sci2_fi    = glob.glob(path_d+"obs002/*_calints.fits")
ref_fi     = glob.glob(path_d+"obs004/*_calints.fits")
sci_bkg_fi = glob.glob(path_d+"obs003/*_calints.fits")
ref_bkg_fi = glob.glob(path_d+"obs005/*_calints.fits")

sci1_fi.sort()
sci2_fi.sort()
ref_fi.sort()
sci_bkg_fi.sort()
ref_bkg_fi.sort()

sci1 = fits.open(sci1_fi[0])[1].data
sci2 = fits.open(sci2_fi[0])[1].data
ref = []
for i in ref_fi: ref.append(fits.open(i)[1].data)
ref = np.array(ref)
sci_bkg = []
ref_bkg = []
for i in sci_bkg_fi : sci_bkg.append(fits.open(i)[1].data)
for i in ref_bkg_fi : ref_bkg.append(fits.open(i)[1].data)
sci_bkg = np.array(sci_bkg)
ref_bkg = np.array(ref_bkg)

print("sci roll 1 : ", np.shape(sci1))
print("sci roll 2 : ", np.shape(sci2))
print("reference  : ", np.shape(ref))
print("bkg sci    : ", np.shape(sci_bkg))
print("bkg ref    : ", np.shape(ref_bkg))

### full data
sci1_ful    = copy.deepcopy(sci1)
sci2_ful    = copy.deepcopy(sci2)
sci_bkg_ful = copy.deepcopy(sci_bkg)

sci1_ful    = np.nanmedian(sci1_ful, axis=0)
sci2_ful    = np.nanmedian(sci2_ful, axis=0)
sci_bkg_ful = np.nanmedian(sci_bkg_ful, axis=1)
ref         = np.nanmedian(ref, axis=1)
ref_bkg     = np.nanmedian(ref_bkg, axis=1)

### master BKG full data
sci_bkg_ful = np.minimum(sci_bkg_ful[0,:,:], sci_bkg_ful[1,:,:])
ref_bkg     = np.minimum(ref_bkg[0,:,:], ref_bkg[1,:,:])

### sigmaclipping + clean NaN + bkg subtraction
sci1_ful = imp.clean_image_nan(imp.remove_bad_pixel(sci1_ful - sci_bkg_ful,3,5))     #science - background
sci2_ful = imp.clean_image_nan(imp.remove_bad_pixel(sci2_ful - sci_bkg_ful,3,5))     #science - background
for i in range(len(ref)): ref[i,:,:] = imp.clean_image_nan(np.array(imp.remove_bad_pixel(ref[i,:,:] - ref_bkg,3,5)))

### Shift and cut
sci1_ful = nd.shift(sci1_ful,[newd/2-y0,newd/2-x0])[0:newd,0:newd]
sci2_ful = nd.shift(sci2_ful,[newd/2-y0,newd/2-x0])[0:newd,0:newd]
tmp      = np.ndarray([9,newd,newd])
for i in range(len(ref)): tmp[i,:,:] = nd.shift(ref[i,:,:],[newd/2-y0,newd/2-x0])[0:newd,0:newd]
ref      = tmp


ref = fits.open(path_s+"cube_lib_ref_1140.fits")[0].data


# PCA full data
PCA1_ful = imp.sub_PCA(sci1_ful, ref, [newd/2,newd/2], 40, dim=100.5, plot=False)
PCA2_ful = imp.sub_PCA(sci2_ful, ref, [newd/2,newd/2], 40, dim=100.5, plot=False)

imp.save_fits(PCA1_ful  , path_s+"TWA7_roll-1_60-calints_PCA.fits")
imp.save_fits(PCA2_ful  , path_s+"TWA7_roll-2_60-calints_PCA.fits")

mask = np.sqrt((X - newd/2)**2 + (Y-newd/2)**2) >= 10
mask *= np.sqrt((X - newd/2)**2 + (Y-newd/2)**2) <= 60

lcomb1_ful = imp.MIRI_linear_comb_mask(sci1_ful, ref, mask, use_minimize=False)
lcomb2_ful = imp.MIRI_linear_comb_mask(sci2_ful, ref, mask, use_minimize=False)

imp.save_fits(lcomb1_ful, path_s+"TWA7_roll-1_60-calints_LComb.fits")
imp.save_fits(lcomb2_ful, path_s+"TWA7_roll-2_60-calints_LComb.fits")


mask_pla1     = np.sqrt((X - newd/2 -2)**2 + (Y - newd/2+13)**2)
mask_pla2     = np.sqrt((X - newd/2 -1)**2 + (Y - newd/2+13)**2)
mask_noise    = np.sqrt((X - newd/2)**2 + (Y - newd/2)**2)
mask_sel1     = (mask_noise >= 13-dr) * (mask_noise < 13+dr) * (mask_pla1>dr)
mask_sel2     = (mask_noise >= 13-dr) * (mask_noise < 13+dr) * (mask_pla2>dr)
mask_pla1     = mask_pla1 < dr
mask_pla2     = mask_pla2 < dr


### Std dev azimutale dans un anneau de 3 L/D de largeur
std_PCA1   = [] 
std_PCA2   = [] 
std_lcomb1 = []
std_lcomb2 = [] 
flx_PCA1   = []
flx_PCA2   = []
flx_lcomb1 = []
flx_lcomb2 = []

std_PCA1.append(np.nanstd(PCA1_ful[mask_sel1]))
std_PCA2.append(np.nanstd(PCA2_ful[mask_sel2])) 
std_lcomb1.append(np.nanstd(lcomb1_ful[mask_sel1])) 
std_lcomb2.append(np.nanstd(lcomb2_ful[mask_sel2]))

flx_PCA1.append(np.nansum(PCA1_ful[mask_pla1]))
flx_PCA2.append(np.nansum(PCA2_ful[mask_pla2]))
flx_lcomb1.append(np.nansum(lcomb1_ful[mask_pla1]))
flx_lcomb2.append(np.nansum(lcomb2_ful[mask_pla2]))

print("PCA 1 full SNR :",std_PCA1,flx_PCA1, flx_PCA1[0]/std_PCA1[0])
print("PCA 2 full SNR :",std_PCA2,flx_PCA2, flx_PCA2[0]/std_PCA2[0])
print("LComb 1 full SNR :",std_lcomb1,flx_lcomb1, flx_lcomb1[0]/std_lcomb1[0])
print("LComb 2 full SNR :",std_lcomb2,flx_lcomb2, flx_lcomb2[0]/std_lcomb2[0])


#corono image
fig1,[[sub1,sub2],[sub3,sub4]] = plt.subplots(nrows=2,ncols=2,figsize=(20,20))
plt.subplots_adjust(wspace=0.12, hspace=0.05, left=0.01, right=0.99)
tmp1 = sub1.imshow(PCA1_ful*mask_sel1,origin='lower',cmap='inferno',interpolation='none',vmin=-1,vmax=1,extent=scale)
tmp2 = sub2.imshow(PCA2_ful*mask_sel2,origin='lower',cmap='inferno',interpolation='none',vmin=-1,vmax=1,extent=scale)
tmp3 = sub3.imshow(lcomb1_ful*mask_pla1,origin='lower',cmap='inferno',interpolation='none',vmin=-1,vmax=1,extent=scale)
tmp4 = sub4.imshow(lcomb2_ful*mask_pla2,origin='lower',cmap='inferno',interpolation='none',vmin=-1,vmax=1,extent=scale)
sub1.set_title('PCA - roll 1')
sub2.set_title('PCA - roll 2')
sub3.set_title('LComb - roll 1')
sub4.set_title('Lcomb - roll 2')
sub1.set_ylabel(r'$\Delta$ DEC [arcsec]')
sub3.set_xlabel(r'$\Delta$ RA [arcsec]')
fig1.savefig(path_s+'zone_photometry.pdf',format='pdf',bbox_inches='tight')
plt.close(fig1)




## loop for different value of ints 
ints_tested = [30,29,25,20,18,15,10,5]

for ints in ints_tested:

    sci1_cut    = copy.deepcopy(sci1[0:ints,:,:])
    sci2_cut    = copy.deepcopy(sci2[0:ints,:,:])
    sci_bkg_cut = copy.deepcopy(sci_bkg[:,0:ints,:,:])

    sci1_cut    = np.nanmedian(sci1_cut, axis=0)
    sci2_cut    = np.nanmedian(sci2_cut, axis=0)
    sci_bkg_cut = np.nanmedian(sci_bkg_cut, axis=1)

    ### master BKG cut
    sci_bkg_cut = np.minimum(sci_bkg_cut[0,:,:], sci_bkg_cut[1,:,:])

    ### sigmaclipping + clean NaN + bkg subtraction
    sci1_cut = imp.clean_image_nan(imp.remove_bad_pixel(sci1_cut - sci_bkg_cut,3,5))     #science - background
    sci2_cut = imp.clean_image_nan(imp.remove_bad_pixel(sci2_cut - sci_bkg_cut,3,5))     #science - background
    
    ### Shift and cut
    sci1_cut    = nd.shift(sci1_cut,[newd/2-y0,newd/2-x0])[0:newd,0:newd]
    sci2_cut    = nd.shift(sci2_cut,[newd/2-y0,newd/2-x0])[0:newd,0:newd]

    # PCA partiel data
    PCA1_cut = imp.sub_PCA(sci1_cut, ref, [newd/2,newd/2], 40, dim=100.5, plot=False)
    PCA2_cut = imp.sub_PCA(sci2_cut, ref, [newd/2,newd/2], 40, dim=100.5, plot=False)

    imp.save_fits(PCA1_cut  , path_s+"TWA7_roll-1_"+str(ints)+"-calints_PCA.fits")
    imp.save_fits(PCA2_cut  , path_s+"TWA7_roll-2_"+str(ints)+"-calints_PCA.fits")

    ### LCOMB partiel data    
    lcomb1_cut = imp.MIRI_linear_comb_mask(sci1_cut, ref, mask, use_minimize=False)
    lcomb2_cut = imp.MIRI_linear_comb_mask(sci2_cut, ref, mask, use_minimize=False)

    imp.save_fits(lcomb1_cut, path_s+"TWA7_roll-1_"+str(ints)+"-calints_LComb.fits")
    imp.save_fits(lcomb2_cut, path_s+"TWA7_roll-2_"+str(ints)+"-calints_LComb.fits")

    std_PCA1.append(np.nanstd(PCA1_cut[mask_sel1])) 
    std_PCA2.append(np.nanstd(PCA2_cut[mask_sel2])) 
    std_lcomb1.append(np.nanstd(lcomb1_cut[mask_sel1])) 
    std_lcomb2.append(np.nanstd(lcomb2_cut[mask_sel2])) 
    flx_PCA1.append(np.nansum(PCA1_cut[mask_pla1]))
    flx_PCA2.append(np.nansum(PCA2_cut[mask_pla2]))
    flx_lcomb1.append(np.nansum(lcomb1_cut[mask_pla1]))
    flx_lcomb2.append(np.nansum(lcomb2_cut[mask_pla2]))


    





std_PCA1 = np.array(std_PCA1)
std_PCA2 = np.array(std_PCA2)
std_lcomb1 = np.array(std_lcomb1)
std_lcomb2 = np.array(std_lcomb2)
flx_PCA1 = np.array(flx_PCA1)
flx_PCA2 = np.array(flx_PCA2)
flx_lcomb1 = np.array(flx_lcomb1)
flx_lcomb2 = np.array(flx_lcomb2)

sci roll 1 :  (60, 224, 288)
sci roll 2 :  (60, 224, 288)
reference  :  (9, 6, 224, 288)
bkg sci    :  (2, 60, 224, 288)
bkg ref    :  (2, 6, 224, 288)


/tmp/ipykernel_33286/3490816624.py:39: RuntimeWarning: All-NaN slice encountered
  sci1_ful    = np.nanmedian(sci1_ful, axis=0)
/tmp/ipykernel_33286/3490816624.py:40: RuntimeWarning: All-NaN slice encountered
  sci2_ful    = np.nanmedian(sci2_ful, axis=0)
/tmp/ipykernel_33286/3490816624.py:41: RuntimeWarning: All-NaN slice encountered
  sci_bkg_ful = np.nanmedian(sci_bkg_ful, axis=1)
/tmp/ipykernel_33286/3490816624.py:42: RuntimeWarning: All-NaN slice encountered
  ref         = np.nanmedian(ref, axis=1)
/tmp/ipykernel_33286/3490816624.py:43: RuntimeWarning: All-NaN slice encountered
  ref_bkg     = np.nanmedian(ref_bkg, axis=1)
/home/cperrot/Work/code/JWST/Module/imodex_miri_postprocess_pipeline.py:250: RuntimeWarning: Mean of empty slice
  function=lambda x: np.nanmean(x[mask.flatten().astype(bool)]),
/home/cperrot/anaconda3/envs/jwst/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtyp

PCA 1 full SNR : [np.float64(0.6759395573151096)] [np.float64(48.75690220550173)] 72.13204446736091
PCA 2 full SNR : [np.float64(0.6901230148872715)] [np.float64(42.244309133501936)] 61.21272327137555
LComb 1 full SNR : [np.float64(0.14361866251178448)] [np.float64(12.808356593404454)] 89.1830934044068
LComb 2 full SNR : [np.float64(0.12541721351479718)] [np.float64(8.233128688239686)] 65.64592257719322


/tmp/ipykernel_33286/3490816624.py:145: RuntimeWarning: All-NaN slice encountered
  sci1_cut    = np.nanmedian(sci1_cut, axis=0)
/tmp/ipykernel_33286/3490816624.py:146: RuntimeWarning: All-NaN slice encountered
  sci2_cut    = np.nanmedian(sci2_cut, axis=0)
/tmp/ipykernel_33286/3490816624.py:147: RuntimeWarning: All-NaN slice encountered
  sci_bkg_cut = np.nanmedian(sci_bkg_cut, axis=1)
/home/cperrot/Work/code/JWST/Module/imodex_miri_postprocess_pipeline.py:250: RuntimeWarning: Mean of empty slice
  function=lambda x: np.nanmean(x[mask.flatten().astype(bool)]),
/home/cperrot/anaconda3/envs/jwst/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [58]:

SNR_PCA1 = flx_PCA1/std_PCA1/np.sqrt(nbpixel)
SNR_PCA2 = flx_PCA2/std_PCA2/np.sqrt(nbpixel)
SNR_lcomb1 = flx_lcomb1/std_lcomb1/np.sqrt(nbpixel)
SNR_lcomb2 = flx_lcomb2/std_lcomb2/np.sqrt(nbpixel)

In [59]:
fig1 = plt.figure(figsize=(10,7))
plt.plot([60]+ints_tested,SNR_PCA1,color="red",label="PCA - roll 1")
plt.plot([60]+ints_tested,SNR_PCA2,'--',color="red",label="PCA - roll 2")
plt.plot([60]+ints_tested,SNR_lcomb1,color="blue",label="LComb - roll 1")
plt.plot([60]+ints_tested,SNR_lcomb2,'--',color="blue",label="LComb - roll 2")
plt.title('SNR with different ints', fontsize=20)
plt.xlabel('Number of ints', fontsize=16)
plt.ylabel('SNR', fontsize=16)
plt.grid(True, which="major", ls="-")
plt.grid(True, which="minor", ls="--")
plt.ylim([0,35])
plt.xlim([0,60])
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(fontsize=14)
fig1.savefig(path_s+'SNR_vs_ints.pdf',format='pdf',bbox_inches='tight')
plt.close(fig1)

In [60]:
#corono image
box=20
fig1,[[sub1,sub2],[sub3,sub4]] = plt.subplots(nrows=2,ncols=2,figsize=(8,10))
plt.subplots_adjust(wspace=0.12, hspace=0.15, left=0.01, right=0.99)
tmp1 = sub1.imshow((PCA1_ful*mask_sel1)[newd//2-box:newd//2+box,newd//2-box:newd//2+box],origin='lower',cmap='inferno',interpolation='none',vmin=-1,vmax=1,extent=scale)
tmp2 = sub2.imshow((PCA2_ful*mask_sel2)[newd//2-box:newd//2+box,newd//2-box:newd//2+box],origin='lower',cmap='inferno',interpolation='none',vmin=-1,vmax=1,extent=scale)
tmp3 = sub3.imshow((PCA1_ful*mask_pla1)[newd//2-box:newd//2+box,newd//2-box:newd//2+box],origin='lower',cmap='inferno',interpolation='none',vmin=-1,vmax=1,extent=scale)
tmp4 = sub4.imshow((PCA2_ful*mask_pla2)[newd//2-box:newd//2+box,newd//2-box:newd//2+box],origin='lower',cmap='inferno',interpolation='none',vmin=-1,vmax=1,extent=scale)
sub1.set_title('PCA - roll 1 - noise mask')
sub2.set_title('PCA - roll 2 - noise mask')
sub3.set_title('PCA - roll 1 - planet mask')
sub4.set_title('PCA - roll 2 - planet mask')
sub1.set_ylabel(r'$\Delta$ DEC [arcsec]')
sub3.set_xlabel(r'$\Delta$ RA [arcsec]')
fig1.savefig(path_s+'zone_photometry_pca.pdf',format='pdf',bbox_inches='tight')
plt.close(fig1)

#corono image
box=20
fig1,[[sub1,sub2],[sub3,sub4]] = plt.subplots(nrows=2,ncols=2,figsize=(8,10))
plt.subplots_adjust(wspace=0.12, hspace=0.15, left=0.01, right=0.99)
tmp1 = sub1.imshow((lcomb1_ful*mask_sel1)[newd//2-box:newd//2+box,newd//2-box:newd//2+box],origin='lower',cmap='inferno',interpolation='none',vmin=-1,vmax=1,extent=scale)
tmp2 = sub2.imshow((lcomb2_ful*mask_sel2)[newd//2-box:newd//2+box,newd//2-box:newd//2+box],origin='lower',cmap='inferno',interpolation='none',vmin=-1,vmax=1,extent=scale)
tmp3 = sub3.imshow((lcomb1_ful*mask_pla1)[newd//2-box:newd//2+box,newd//2-box:newd//2+box],origin='lower',cmap='inferno',interpolation='none',vmin=-1,vmax=1,extent=scale)
tmp4 = sub4.imshow((lcomb2_ful*mask_pla2)[newd//2-box:newd//2+box,newd//2-box:newd//2+box],origin='lower',cmap='inferno',interpolation='none',vmin=-1,vmax=1,extent=scale)
sub1.set_title('LComb - roll 1 - noise mask')
sub2.set_title('LComb - roll 2 - noise mask')
sub3.set_title('LComb - roll 1 - planet mask')
sub4.set_title('LComb - roll 2 - planet mask')
sub1.set_ylabel(r'$\Delta$ DEC [arcsec]')
sub3.set_xlabel(r'$\Delta$ RA [arcsec]')
fig1.savefig(path_s+'zone_photometry_lcomb.pdf',format='pdf',bbox_inches='tight')
plt.close(fig1)